In [ ]:
!pip install -q -U datasets pandas scikit-learn transformers librosa

import pandas as pd
import numpy as np
import librosa
import json
from datasets import load_dataset, Dataset, Audio, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

In [ ]:
# ==========================================
# CENTRALIZED DATA PIPELINE CONFIGURATION
# ==========================================
class DataConfig:
    # --- API & Authentication ---
    HF_TOKEN = "YOUR_HUGGINGFACE_TOKEN" # Required for gated models like Llama-3.1

    # --- Module A: Text Pipeline (Hindi-to-Haryanvi) ---
    TEXT_DATASET_PATH = "synthetic_haryanvi_corpus.json"
    TEXT_MIN_WORDS = 3
    TEXT_MAX_WORDS = 25
    LLM_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
    LLM_MAX_LENGTH = 128  # Keeps memory footprint low for 8GB VRAM limits

    # --- Module B: Audio Pipeline (TTS) ---
    TTS_DATASET_ID = "ankitdhiman/haryanvi-tts"
    TARGET_SR = 22050     # Standard for VITS
    AUDIO_MIN_DURATION = 1.0  # seconds
    AUDIO_MAX_DURATION = 10.0 # seconds

    # --- Splitting & Reproducibility ---
    RANDOM_SEED = 42
    TEXT_TEST_SPLIT = 0.20  # Leaves 80% for train, 20% for val/test
    TEXT_VAL_SPLIT = 0.50   # Splits the 20% equally into 10% val, 10% test
    AUDIO_EVAL_SPLIT = 0.10 # 90/10 split for VITS training

config = DataConfig()

In [ ]:
# ==========================================
# MODULE A: TEXT DATA INGESTION & QUALITY ASSESSMENT
# ==========================================
print("--- Loading Text Dataset ---")

# Placeholder df for now
df_text = pd.DataFrame({
    "hindi": ["मैं घर नहीं जा रहा हूँ।", "हम कल बाजार जाएंगे।"],
    "haryanvi": ["मैं घर कोनी जा रया सूं।", "म्हारे काल बजार जावेंगे।"]
})
print(f"Initial synthetic dataset size: {len(df_text)}")

print("\n--- Quality Assessment ---")
missing_values = df_text.isnull().sum()
print(f"Missing values:\n{missing_values}")

initial_len = len(df_text)
df_text = df_text.drop_duplicates()
print(f"Removed {initial_len - len(df_text)} duplicate rows to prevent leakage.")

# Filter out extreme outliers to maintain quality
df_text['hindi_len'] = df_text['hindi'].apply(lambda x: len(x.split()))
df_text_clean = df_text[
    (df_text['hindi_len'] >= config.TEXT_MIN_WORDS) &
    (df_text['hindi_len'] <= config.TEXT_MAX_WORDS)
].copy()

# ==========================================
# TRAIN / VAL / TEST SPLIT
# ==========================================
train_df, temp_df = train_test_split(
    df_text_clean,
    test_size=config.TEXT_TEST_SPLIT,
    random_state=config.RANDOM_SEED
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=config.TEXT_VAL_SPLIT,
    random_state=config.RANDOM_SEED
)

print(f"\n--- Split Sizes ---")
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

text_dataset_dict = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False)
})

In [ ]:
# ==========================================
# LLM FORMATTING & TOKENIZATION PIPELINE
# ==========================================
tokenizer = AutoTokenizer.from_pretrained(config.LLM_MODEL_ID, token=config.HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

def format_for_llama(examples):
    formatted_texts = []
    for hindi, haryanvi in zip(examples['hindi'], examples['haryanvi']):
        dialog = [
            {"role": "system", "content": "You are a precise Hindi to Haryanvi (Bangru) translator. Convert the standard Hindi input into grammatically correct Haryanvi. Follow systematic lexical substitutions and morpho-syntactic rules."},
            {"role": "user", "content": hindi},
            {"role": "assistant", "content": haryanvi}
        ]
        formatted_texts.append(tokenizer.apply_chat_template(dialog, tokenize=False))

    tokenized = tokenizer(
        formatted_texts,
        max_length=config.LLM_MAX_LENGTH,
        padding="max_length",
        truncation=True
    )
    return tokenized

llm_ready_dataset = text_dataset_dict.map(
    format_for_llama,
    batched=True,
    remove_columns=['hindi', 'haryanvi', 'hindi_len']
)
print("\nLLM Dataset properly formatted with Chat Templates.")

In [ ]:
# ==========================================
# MODULE B: AUDIO DATASET (TTS) PREPARATION
# ==========================================
print(f"Loading {config.TTS_DATASET_ID}...")
audio_dataset = load_dataset(config.TTS_DATASET_ID, split="train", token=config.HF_TOKEN)

# Force resampling to maintain consistency
audio_dataset = audio_dataset.cast_column("audio", Audio(sampling_rate=config.TARGET_SR))

def calculate_audio_duration(example):
    audio_array = example['audio']['array']
    sr = example['audio']['sampling_rate']
    example['duration'] = len(audio_array) / sr
    return example

audio_dataset = audio_dataset.map(calculate_audio_duration)

# Filter out extreme outliers to prevent VRAM spikes and bad alignment
filtered_audio = audio_dataset.filter(
    lambda x: config.AUDIO_MIN_DURATION <= x['duration'] <= config.AUDIO_MAX_DURATION
)
print(f"Retained {len(filtered_audio)} audio samples after duration filtering.")

# Train / Val Split for Audio
audio_dataset_dict = filtered_audio.train_test_split(
    test_size=config.AUDIO_EVAL_SPLIT,
    seed=config.RANDOM_SEED
)

def prepare_vits_dataset(examples):
    return {
        "audio_array": [audio["array"] for audio in examples["audio"]],
        "normalized_text": [text.strip() for text in examples["transcript"]]
    }

final_tts_dataset = audio_dataset_dict.map(
    prepare_vits_dataset,
    batched=True,
    remove_columns=audio_dataset_dict['train'].column_names
)

print("\n--- Audio Pipeline Ready ---")
print(f"Audio Train Split: {len(final_tts_dataset['train'])} samples")
print(f"Audio Eval Split: {len(final_tts_dataset['test'])} samples")